In [1]:
print("test")

test


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
import logging
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import sklearn
import logging
from sklearn.model_selection import train_test_split
import os
import shutil
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pickle
import optuna
import numpy as np
import pandas as pd
import pickle
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor
from tqdm.auto import tqdm
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

import xgboost as xgb
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

c:\Users\wporc\GitHub\Immo-Eliza-Machine-Learning\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
train_residential_apartment=pd.read_csv(r"datasets\train_residential_apartment.csv")
test_residential_apartment=pd.read_csv(r"datasets\test_residential_apartment.csv")
train_residential_house=pd.read_csv(r"datasets\train_residential_house.csv")
test_residential_house=pd.read_csv(r"datasets\test_residential_house.csv")


In [ ]:
import optuna
import numpy as np
import pandas as pd
import pickle
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor
from tqdm.auto import tqdm


def prepare_data(train_df, test_df):
    train_df = train_df[train_df["price"] > 0].copy()
    test_df = test_df[test_df["price"] > 0].copy()

    train_df["log_price"] = np.log(train_df["price"])
    test_df["log_price"] = np.log(test_df["price"])

    X_train = train_df.drop(columns=["price", "log_price"])
    y_train = train_df["log_price"]

    X_test = test_df.drop(columns=["price", "log_price"])
    y_test = test_df["price"]

    cat_features = X_train.select_dtypes(include=["object"]).columns.tolist()
    return X_train, y_train, X_test, y_test, cat_features


GLOBAL_BEST_RMSE = float("inf")
GLOBAL_BEST_MODEL_PATH = ""


def objective_cat(trial, train_df, test_df, save_name):
    global GLOBAL_BEST_RMSE
    global GLOBAL_BEST_MODEL_PATH

    X_train, y_train, X_test, y_test, cat_features = prepare_data(train_df, test_df)

    params = {
        "depth": trial.suggest_int("depth", 6, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 2, 25),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.1, 10),
        "random_strength": trial.suggest_float("random_strength", 0.5, 5),
        "iterations": trial.suggest_int("iterations", 1500, 5000),
        "loss_function": "RMSE",
        "verbose": 0
    }

    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train, cat_features=cat_features, verbose=0)

    preds = np.exp(model.predict(X_test))
    rmse = float(np.sqrt(mean_squared_error(y_test, preds)))

    if rmse < GLOBAL_BEST_RMSE:
        GLOBAL_BEST_RMSE = rmse
        GLOBAL_BEST_MODEL_PATH = f"models_optimisation/{save_name}_BEST.pkl"
        pickle.dump(model, open(GLOBAL_BEST_MODEL_PATH, "wb"))
        print(f"[AUTO-SAVE] New best RMSE={rmse:.3f} → {GLOBAL_BEST_MODEL_PATH}")

    return rmse


def tune_catboost(train_df, test_df, name, n_trials=50, storage_name="cat_study"):
    global GLOBAL_BEST_RMSE, GLOBAL_BEST_MODEL_PATH
    GLOBAL_BEST_RMSE = float("inf")
    GLOBAL_BEST_MODEL_PATH = ""

    study = optuna.create_study(
        direction="minimize",
        study_name=storage_name,
        storage=f"sqlite:///{storage_name}.db",
        load_if_exists=True
    )

    for _ in tqdm(range(n_trials), desc=f"Tuning CAT {name}", ncols=100):
        trial = study.ask()
        value = objective_cat(trial, train_df, test_df, name)
        study.tell(trial, value)

    print("Best params:", study.best_params)
    print("Best RMSE:", study.best_value)
    print("Best model:", GLOBAL_BEST_MODEL_PATH)
    return GLOBAL_BEST_MODEL_PATH


In [ ]:
best_model = tune_catboost(
    train_residential_apartment,
    test_residential_apartment,
    name="cat_res_apt",
    n_trials=50,
    storage_name="cat_res_apt"
)


In [ ]:
best_model = tune_catboost(
    train_residential_house,
    test_residential_house,
    name="cat_res_house",
    n_trials=50,
    storage_name="cat_res_house"
)


In [14]:
import optuna
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

import xgboost as xgb


# ============================================
#  PREPROCESSOR 
# ============================================
def build_preprocessor(df):
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    cat_cols = df.select_dtypes(include=["object"]).columns

    preprocess = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ])

    return preprocess


# ============================================
#  PREPARE DATA
# ============================================
def prepare_data(train_df, test_df):
    train_df = train_df[train_df["price"] > 0].copy()
    test_df = test_df[test_df["price"] > 0].copy()

    train_df["log_price"] = np.log(train_df["price"])
    test_df["log_price"] = np.log(test_df["price"])

    X_train = train_df.drop(columns=["price", "log_price"])
    y_train = train_df["log_price"]

    X_test = test_df.drop(columns=["price", "log_price"])
    y_test = test_df["price"]  # real price (since we do exp())

    return X_train, y_train, X_test, y_test


# ============================================
#  XGB OBJECTIVE 
# ============================================
GLOBAL_BEST_RMSE = float("inf")
GLOBAL_BEST_MODEL_PATH = None


def objective_xgb(trial, train_df, test_df, save_name):
    global GLOBAL_BEST_RMSE
    global GLOBAL_BEST_MODEL_PATH

    X_train, y_train, X_test, y_test = prepare_data(train_df, test_df)

    # hiperparameters
    params = {
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.25),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "n_estimators": trial.suggest_int("n_estimators", 300, 2000),
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "verbosity": 0,
    }

    preprocess = build_preprocessor(X_train)

    model = Pipeline([
        ("preprocess", preprocess),
        ("model", xgb.XGBRegressor(**params))
    ])

    model.fit(X_train, y_train)

    preds = np.exp(model.predict(X_test))
    rmse = float(np.sqrt(mean_squared_error(y_test, preds)))

    # autosave
    if rmse < GLOBAL_BEST_RMSE:
        GLOBAL_BEST_RMSE = rmse
        GLOBAL_BEST_MODEL_PATH = f"models_optimisation/{save_name}_BEST.pkl"
        pickle.dump(model, open(GLOBAL_BEST_MODEL_PATH, "wb"))
        print(f"[AUTO-SAVE] New BEST XGB RMSE={rmse:.3f} -> saved {GLOBAL_BEST_MODEL_PATH}")

    return rmse


# ============================================
#  MAIN TUNING FUNCTION
# ============================================
def tune_xgb(train_df, test_df, name, n_trials=50, storage_name="xgb_study"):
    global GLOBAL_BEST_RMSE, GLOBAL_BEST_MODEL_PATH
    GLOBAL_BEST_RMSE = float("inf")

    study = optuna.create_study(
        direction="minimize",
        study_name=storage_name,
        storage=f"sqlite:///{storage_name}.db",
        load_if_exists=True
    )

    for _ in tqdm(range(n_trials), desc=f"Tuning XGB {name}", ncols=100):
        trial = study.ask()
        value = objective_xgb(trial, train_df, test_df, name)
        study.tell(trial, value)

    print("\nBest params:", study.best_params)
    print("Best RMSE:", study.best_value)
    print("BEST MODEL PATH:", GLOBAL_BEST_MODEL_PATH)

    return GLOBAL_BEST_MODEL_PATH


In [15]:
best_model = tune_xgb(
    train_residential_apartment,
    test_residential_apartment,
    name="xgb_res_apt",
    n_trials=50,
    storage_name="xgb_res_apt"
)


[I 2025-12-04 10:53:38,157] A new study created in RDB with name: xgb_res_apt
Tuning XGB xgb_res_apt:   2%|▊                                       | 1/50 [00:02<01:42,  2.09s/it]

[AUTO-SAVE] New BEST XGB RMSE=62134.051 -> saved models_optimisation/xgb_res_apt_BEST.pkl


Tuning XGB xgb_res_apt:   4%|█▌                                      | 2/50 [00:03<01:10,  1.46s/it]

[AUTO-SAVE] New BEST XGB RMSE=62060.731 -> saved models_optimisation/xgb_res_apt_BEST.pkl


Tuning XGB xgb_res_apt:   6%|██▍                                     | 3/50 [00:04<01:14,  1.59s/it]

[AUTO-SAVE] New BEST XGB RMSE=57938.071 -> saved models_optimisation/xgb_res_apt_BEST.pkl


Tuning XGB xgb_res_apt:  10%|████                                    | 5/50 [00:09<01:25,  1.90s/it]


KeyboardInterrupt: 

In [ ]:
best_model = tune_xgb(
    train_residential_house,
    test_residential_house,
    name="xgb_res_house",
    n_trials=50,
    storage_name="xgb_res_house"
)


In [ ]:


# =====================================================
#  PREPROCESSOR  
# =====================================================
def build_preprocessor(df):
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    cat_cols = df.select_dtypes(include=["object"]).columns

    preprocess = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),

        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ])

    return preprocess


# =====================================================
#  DATA SPLIT + TARGET LOG TRANSFORM
# =====================================================
def prepare_data(train_df, test_df):
    train_df = train_df[train_df["price"] > 0].copy()
    test_df = test_df[test_df["price"] > 0].copy()

    train_df["log_price"] = np.log(train_df["price"])
    test_df["log_price"] = np.log(test_df["price"])

    X_train = train_df.drop(columns=["price", "log_price"])
    y_train = train_df["log_price"]

    X_test = test_df.drop(columns=["price", "log_price"])
    y_test = test_df["price"]  # real prices → exp(pred)

    return X_train, y_train, X_test, y_test


# =====================================================
#  GLOBALS FOR AUTOSAVE
# =====================================================
GLOBAL_BEST_RMSE = float("inf")
GLOBAL_BEST_MODEL_PATH = None


# =====================================================
#  LIGHTGBM OBJECTIVE (pipeline-based)
# =====================================================
def objective_lgbm(trial, train_df, test_df, save_name):
    global GLOBAL_BEST_RMSE
    global GLOBAL_BEST_MODEL_PATH

    X_train, y_train, X_test, y_test = prepare_data(train_df, test_df)

    params = {
        "objective": "regression",
        "metric": "rmse",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 120),
        "verbosity": -1,
        "force_col_wise": True,   
    }

    preprocess = build_preprocessor(X_train)

    model = Pipeline([
        ("preprocess", preprocess),
        ("model", lgb.LGBMRegressor(**params))
    ])

    model.fit(X_train, y_train)

    preds = np.exp(model.predict(X_test))
    rmse = float(np.sqrt(mean_squared_error(y_test, preds)))

    # autosave best model
    if rmse < GLOBAL_BEST_RMSE:
        GLOBAL_BEST_RMSE = rmse
        GLOBAL_BEST_MODEL_PATH = f"models_optimisation/{save_name}_BEST.pkl"
        pickle.dump(model, open(GLOBAL_BEST_MODEL_PATH, "wb"))
        print(f"[AUTO-SAVE] New BEST LGBM RMSE={rmse:.3f}  Saved -> {GLOBAL_BEST_MODEL_PATH}")

    return rmse


# =====================================================
#  MAIN TUNING LOOP
# =====================================================
def tune_lgbm(train_df, test_df, name, n_trials=50, storage_name="lgbm_study"):
    global GLOBAL_BEST_RMSE, GLOBAL_BEST_MODEL_PATH
    GLOBAL_BEST_RMSE = float("inf")

    study = optuna.create_study(
        direction="minimize",
        study_name=storage_name,
        storage=f"sqlite:///{storage_name}.db",
        load_if_exists=True
    )

    for _ in tqdm(range(n_trials), desc=f"Tuning LGBM {name}", ncols=100):
        trial = study.ask()
        score = objective_lgbm(trial, train_df, test_df, name)
        study.tell(trial, score)

    print("\nBest params:", study.best_params)
    print("Best RMSE:", study.best_value)
    print("BEST MODEL PATH:", GLOBAL_BEST_MODEL_PATH)

    return GLOBAL_BEST_MODEL_PATH


In [13]:
best_model = tune_lgbm(
    train_residential_apartment,
    test_residential_apartment,
    name="lgbm_res_apt",
    n_trials=50,
    storage_name="lgbm_res_apt"
)


[I 2025-12-04 10:52:48,008] Using an existing study with name 'lgbm_res_apt' instead of creating a new one.
Tuning LGBM lgbm_res_apt:   0%|                                              | 0/50 [00:00<?, ?it/s]c:\Users\wporc\GitHub\Immo-Eliza-Machine-Learning\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
Tuning LGBM lgbm_res_apt:   2%|▊                                     | 1/50 [00:01<01:13,  1.49s/it]

[AUTO-SAVE] New BEST LGBM RMSE=60104.292  Saved -> models_optimisation/lgbm_res_apt_BEST.pkl


c:\Users\wporc\GitHub\Immo-Eliza-Machine-Learning\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
Tuning LGBM lgbm_res_apt:   4%|█▌                                    | 2/50 [00:03<01:15,  1.57s/it]c:\Users\wporc\GitHub\Immo-Eliza-Machine-Learning\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
Tuning LGBM lgbm_res_apt:   6%|██▎                                   | 3/50 [00:04<01:18,  1.66s/it]


KeyboardInterrupt: 

In [ ]:
best_model = tune_lgbm(
    train_residential_house,
    test_residential_house,
    name="lgbm_res_house",
    n_trials=50,
    storage_name="lgbm_res_house"
)
